
# AGN SMBH growth track: dormant → merger → QSO → fading

The growth of a supermassive black hole (SMBH) traces a path through
the (M_BH, L_bol) plane. Starting as a dormant low-mass hole, accretion
during mergers builds both mass and luminosity. Peak luminosity occurs
as a luminous QSO before accretion slows and the system fades. This
example traces four key evolutionary stages and plots both the track
on the (M_BH, L_bol) diagram and the corresponding SEDs.

The composable AGN model allows independent variation of black-hole mass
(``agn_log_mbh``) and bolometric luminosity (``agn_log_lbol``), tracing
how the big blue bump temperature evolves with the SMBH mass along each
stage.

References:

- Soltan, A. 1982, MNRAS, 200, 115 — AGN mass density evolution
- King, A. 2003, ApJ, 596, L27 — AGN feedback on galaxy growth


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

# Minimum SSP for composable AGN (bare-stellar required by Cue nebular backend).
ssp = tengri.load_ssp()

# Build reusable model with AGN composable and all components fixed.
# This allows independent sweeps of log_mbh and log_lbol.
model = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "dpl",
        "all_params": tengri.FIXED,
        "tau_gyr": 2.0,
        "log_total_mass": 10.0,
        "alpha": 2.0,
        "beta": 2.5,
    },
    dust={"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.1, "tau_bc": 0.1},
    agn={
        "type": "composable",
        "disc": {"type": "multicolor", "all_params": tengri.FIXED},
        "all_params": tengri.FIXED,
        "agn_lum_ratio": 1.0,  # turn the composable AGN on (default 0.0 zeros it)
        "agn_log_ledd": -1.0,
        "agn_log_mbh": tengri.Uniform(5.0, 10.0),
        "agn_log_lbol": tengri.Uniform(8.0, 14.0),
    },
    redshift=tengri.Fixed(0.05),
)

# Define four evolutionary stages:
# (label, log_mbh, log_lbol, color, marker, markersize)
STAGES = [
    ("Dormant", 6.0, 9.0, "#2166ac", "o", 25),  # Low mass, low Edd
    ("Merger accreting", 7.0, 11.0, "#f4a582", "s", 20),  # Intermediate
    ("QSO peak", 9.0, 13.0, "#d6604d", "*", 50),  # High mass, high Edd
    ("Fading", 9.0, 11.0, "#92c5de", "D", 18),  # Same mass, lower lbol
]

# Eddington luminosity: L_Edd = 3.2e4 * (M / M_sun) * L_sun
# Eddington ratio λ = L_bol / L_Edd
log_ledd_coefficient = np.log10(3.2e4)

# Sample baseline parameters once
baseline = dict(model.spec.sample(jax.random.PRNGKey(0)))

# Create a 2×2 layout: top row is (M_BH, L_bol) diagram with track,
# bottom row shows SEDs for each stage
fig = plt.figure(figsize=(9.0, 8.5))
gs = fig.add_gridspec(2, 2, hspace=0.40, wspace=0.30, left=0.10, right=0.95, top=0.95, bottom=0.08)
ax_track = fig.add_subplot(gs[0, :])  # Full width for trajectory plot
ax_seds = [
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[1, 1]),
]

# ============================================================================
# Top: (M_BH, L_bol) trajectory with Eddington limit
# ============================================================================

# Plot Eddington limit: L_bol / L_sun = lambda * 3.2e4 * M / M_sun
# At lambda = 1 (on the limit), log_lbol = log_mbh + log(3.2e4)
mbh_plot = np.logspace(6, 10, 100)
lbol_edd_lambda1 = mbh_plot * 3.2e4
ax_track.fill_between(
    mbh_plot,
    lbol_edd_lambda1,
    lbol_edd_lambda1 * 10,
    alpha=0.15,
    color="red",
    label="Super-Eddington (L > L_Edd)",
)
ax_track.loglog(
    mbh_plot,
    lbol_edd_lambda1,
    "r-",
    lw=1.5,
    alpha=0.5,
    label=r"Eddington limit (L = L$_{\mathrm{Edd}}$)",
)

# Plot growth track: connect stages with arrows
for _i, (label, log_mbh, log_lbol, color, marker, msize) in enumerate(STAGES):
    mbh = 10.0**log_mbh
    lbol = 10.0**log_lbol
    ax_track.plot(
        mbh,
        lbol,
        marker=marker,
        markersize=msize,
        color=color,
        mec="black",
        mew=1.0,
        zorder=5,
        label=label,
    )

# Draw arrows connecting consecutive stages to show evolution path
for i in range(len(STAGES) - 1):
    _, log_mbh_i, log_lbol_i, _, _, _ = STAGES[i]
    _, log_mbh_j, log_lbol_j, _, _, _ = STAGES[i + 1]
    mbh_i, lbol_i = 10.0**log_mbh_i, 10.0**log_lbol_i
    mbh_j, lbol_j = 10.0**log_mbh_j, 10.0**log_lbol_j
    # Draw arrow with slight offset to avoid overlap with markers
    ax_track.arrow(
        mbh_i * 1.1,
        lbol_i,
        (mbh_j - mbh_i) * 0.8,
        (lbol_j - lbol_i) * 0.8,
        head_width=0.08,
        head_length=0.05,
        fc="gray",
        ec="gray",
        alpha=0.4,
        length_includes_head=True,
    )

# Annotations for physics context
ax_track.text(
    3e6,
    3e11,
    "Sub-Eddington",
    fontsize=10,
    color="0.5",
    rotation=0,
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=2),
    zorder=10,
)

ax_track.set_xlim(1e5, 1e10)
ax_track.set_ylim(5e7, 1e14)
ax_track.set_xlabel(r"Black hole mass $M_{\mathrm{BH}}$ [$M_\odot$]")
ax_track.set_ylabel(r"Bolometric luminosity $L_{\mathrm{bol}}$ [$L_\odot$]")
ax_track.legend(
    loc="upper left", frameon=True, fontsize=9, framealpha=0.9, ncol=3, columnspacing=0.5
)
ax_track.grid(True, alpha=0.2, which="both")

# ============================================================================
# Bottom: SEDs for selected stages (2 left, 2 right)
# ============================================================================

C_AA_PER_S = 2.998e18

# Top-left: Dormant + Merger
for ax_idx, stages_pair in enumerate([(STAGES[0], STAGES[1]), (STAGES[2], STAGES[3])]):
    ax = ax_seds[ax_idx]
    for _stage_idx, (label, log_mbh, log_lbol, color, _marker, _msize) in enumerate(stages_pair):
        params = {
            **baseline,
            "agn_log_mbh": jnp.float64(log_mbh),
            "agn_log_lbol": jnp.float64(log_lbol),
        }
        out = model.predict(params)
        wave = np.asarray(model.wavelengths)
        nu = C_AA_PER_S / wave  # frequency in Hz
        nu_l_nu = nu * np.asarray(out.rest_sed())

        ax.loglog(wave, nu_l_nu, color=color, lw=1.5, label=label)

    ax.set_xlim(100, 1e5)
    ax.set_ylim(1e40, 1e48)
    ax.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]")
    ax.set_ylabel(r"$\nu L_\nu$  [erg s$^{-1}$]")
    ax.legend(frameon=True, fontsize=8)
    ax.grid(True, alpha=0.2, which="both")

plt.savefig("plot_smbh_growth_track.png", dpi=150, bbox_inches="tight")